In [ ]:
# Colab setup
import sys
if "google.colab" in sys.modules:
    %pip install -q pyedautils ephem statsmodels plotly kaleido ipywidgets

import plotly.io as pio
pio.renderers.default = "notebook_connected"

# Long-Term Decomposition

```{admonition} Goal
:class: tip
Decompose multi-year heating data into trend, seasonal, and residual components to understand underlying patterns in energy consumption.
```

```{admonition} Data Basis
:class: note
Central heating power measurements (`centralHeating.csv`) recorded over multiple years.
```

## Data Preparation

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/retomarek/edap/main/edap/sampleData/centralHeating.csv"
df = pd.read_csv(url, sep=";")
df["time"] = pd.to_datetime(df["time"])
df = df.set_index("time")

# energyHeatingMeter is cumulative → daily consumption = diff of daily max
daily = df[["energyHeatingMeter"]].resample("D").max().diff().dropna()
daily = daily.reset_index()
daily.columns = ["timestamp", "value"]
daily = daily[daily["value"] >= 0]  # remove negative artifacts
daily.head()

## Visualization

In [ ]:
from pyedautils.plots.stat import plot_decomposition

fig = plot_decomposition(daily, title="Heating Demand — Long-term Decomposition", ylab="Heating [kWh/d]")
fig.show()

```{dropdown} Customization
The `plot_decomposition` function accepts additional parameters to fine-tune the decomposition:

- **`period`** — The period of the seasonal component (e.g., 365 for daily data with yearly seasonality). If not set, the function attempts to detect it automatically.
- **`s_window`** — The span of the LOESS window for seasonal extraction. A larger value produces a smoother seasonal component.
```